# Parameter Golf — Setup & Pull from GitHub

Run this notebook first to set up the environment on Colab.
It clones both the official repo (for data/baseline) and our aux losses repo.

**Usage:**
1. Upload this notebook to Google Drive
2. Open in Colab (GPU runtime)
3. Run all cells
4. Then open `parameter_golf_experiments.ipynb` from the cloned repo

## 1. Mount Google Drive (persistent storage for logs/results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Persistent storage on Drive for logs and results
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print(f'Drive directory: {DRIVE_DIR}')

## 2. Clone Official Parameter Golf Repo (data + baseline)

In [ ]:
%cd /content

# Clone official repo if not already present
if not os.path.exists('/content/parameter-golf'):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
else:
    %cd /content/parameter-golf
    !git pull
    %cd /content

print('Official repo ready')

## 3. Clone Our Auxiliary Losses Repo

In [ ]:
%cd /content

# Clone our aux losses repo if not already present
if not os.path.exists('/content/parameter-golf-aux'):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git
else:
    %cd /content/parameter-golf-aux
    !git pull
    %cd /content

print('Aux losses repo ready')

## 4. Copy Our Files into the Official Repo

In [ ]:
import shutil

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'

# Copy our training scripts
shutil.copy2(f'{AUX}/train_gpt_aux.py', f'{OFFICIAL}/train_gpt_aux.py')
shutil.copy2(f'{AUX}/train_gpt_sota.py', f'{OFFICIAL}/train_gpt_sota.py')

# Copy aux_losses module
if os.path.exists(f'{OFFICIAL}/aux_losses'):
    shutil.rmtree(f'{OFFICIAL}/aux_losses')
shutil.copytree(f'{AUX}/aux_losses', f'{OFFICIAL}/aux_losses')

# Symlink logs to Drive for persistence
logs_dir = f'{OFFICIAL}/logs'
if os.path.exists(logs_dir) and not os.path.islink(logs_dir):
    # Move any existing logs to Drive first
    for f in os.listdir(logs_dir):
        shutil.copy2(f'{logs_dir}/{f}', f'{DRIVE_DIR}/logs/{f}')
    shutil.rmtree(logs_dir)
if not os.path.exists(logs_dir):
    os.symlink(f'{DRIVE_DIR}/logs', logs_dir)

print('Files copied into official repo')
print(f'Logs will persist at: {DRIVE_DIR}/logs/')

## 5. Install Dependencies & Download Data

In [ ]:
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

In [ ]:
%cd /content/parameter-golf

# Download 1 shard for quick experiments (use --train-shards 10 for fuller runs)
SHARDS = 1
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards {SHARDS}

## 6. Verify Everything Works

In [ ]:
%cd /content/parameter-golf

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()

# Syntax check
!python3 -c "import py_compile; py_compile.compile('train_gpt_aux.py', doraise=True); print('train_gpt_aux.py: OK')"
!python3 -c "from aux_losses import focal_cross_entropy, inter_layer_decorrelation_loss, representation_rank_loss, unigram_kl_loss; print('aux_losses imports: OK')"

# Check data
import glob
train_files = glob.glob('data/datasets/fineweb10B_sp1024/fineweb_train_*.bin')
val_files = glob.glob('data/datasets/fineweb10B_sp1024/fineweb_val_*.bin')
print(f'Train shards: {len(train_files)}')
print(f'Val shards: {len(val_files)}')
print()
print('Setup complete. Ready to run experiments.')

## 7. Quick Pull (run this cell to update code without re-running everything)

After pushing changes from WSL, run just this cell to pull the latest code.

In [ ]:
import shutil, os

# Pull latest from our repo
%cd /content/parameter-golf-aux
!git pull

# Re-copy into official repo
OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'

shutil.copy2(f'{AUX}/train_gpt_aux.py', f'{OFFICIAL}/train_gpt_aux.py')
shutil.copy2(f'{AUX}/train_gpt_sota.py', f'{OFFICIAL}/train_gpt_sota.py')
if os.path.exists(f'{OFFICIAL}/aux_losses'):
    shutil.rmtree(f'{OFFICIAL}/aux_losses')
shutil.copytree(f'{AUX}/aux_losses', f'{OFFICIAL}/aux_losses')

%cd /content/parameter-golf
!python3 -c "import py_compile; py_compile.compile('train_gpt_aux.py', doraise=True); print('Updated & verified OK')"

---

## Experiments

### Baseline (SOTA, no aux losses)

In [ ]:
%cd /content/parameter-golf

!RUN_ID=baseline_seed42 SEED=42 \
 ITERATIONS=500 TRAIN_BATCH_TOKENS=65536 VAL_LOSS_EVERY=100 \
 MAX_WALLCLOCK_SECONDS=300 USE_AUX_LOSSES=0 \
 python3 train_gpt_aux.py

### Focal Loss (gamma=2.0)

In [ ]:
%cd /content/parameter-golf

!RUN_ID=focal_g2_seed42 SEED=42 \
 ITERATIONS=500 TRAIN_BATCH_TOKENS=65536 VAL_LOSS_EVERY=100 \
 MAX_WALLCLOCK_SECONDS=300 \
 USE_AUX_LOSSES=1 USE_FOCAL_LOSS=1 FOCAL_GAMMA=2.0 \
 LAMBDA_DECORR=0 LAMBDA_RANK=0 LAMBDA_UNIGRAM=0 \
 python3 train_gpt_aux.py

### Inter-Layer Decorrelation (lambda=0.01)

In [ ]:
%cd /content/parameter-golf

!RUN_ID=decorr_001_seed42 SEED=42 \
 ITERATIONS=500 TRAIN_BATCH_TOKENS=65536 VAL_LOSS_EVERY=100 \
 MAX_WALLCLOCK_SECONDS=300 \
 USE_AUX_LOSSES=1 USE_FOCAL_LOSS=0 \
 LAMBDA_DECORR=0.01 LAMBDA_RANK=0 LAMBDA_UNIGRAM=0 \
 python3 train_gpt_aux.py

### Representation Rank (lambda=0.05)

In [ ]:
%cd /content/parameter-golf

!RUN_ID=rank_005_seed42 SEED=42 \
 ITERATIONS=500 TRAIN_BATCH_TOKENS=65536 VAL_LOSS_EVERY=100 \
 MAX_WALLCLOCK_SECONDS=300 \
 USE_AUX_LOSSES=1 USE_FOCAL_LOSS=0 \
 LAMBDA_DECORR=0 LAMBDA_RANK=0.05 RANK_EVERY=10 LAMBDA_UNIGRAM=0 \
 python3 train_gpt_aux.py

### Unigram KL (lambda=0.1)

In [ ]:
%cd /content/parameter-golf

!RUN_ID=unigram_01_seed42 SEED=42 \
 ITERATIONS=500 TRAIN_BATCH_TOKENS=65536 VAL_LOSS_EVERY=100 \
 MAX_WALLCLOCK_SECONDS=300 \
 USE_AUX_LOSSES=1 USE_FOCAL_LOSS=0 \
 LAMBDA_DECORR=0 LAMBDA_RANK=0 LAMBDA_UNIGRAM=0.1 \
 python3 train_gpt_aux.py

### Combined (fill in best lambdas after individual tests)

In [ ]:
# Uncomment and fill in best lambdas after individual testing
# %cd /content/parameter-golf
#
# !RUN_ID=combined_seed42 SEED=42 \
#  ITERATIONS=500 TRAIN_BATCH_TOKENS=65536 VAL_LOSS_EVERY=100 \
#  MAX_WALLCLOCK_SECONDS=300 \
#  USE_AUX_LOSSES=1 USE_FOCAL_LOSS=1 FOCAL_GAMMA=2.0 \
#  LAMBDA_DECORR=0.01 LAMBDA_RANK=0.05 LAMBDA_UNIGRAM=0.1 \
#  python3 train_gpt_aux.py

### Collect Results

In [ ]:
import glob, re, shutil

DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

results = []
for logfile in sorted(glob.glob('/content/parameter-golf/logs/*.txt')):
    name = logfile.split('/')[-1].replace('.txt', '')
    with open(logfile) as f:
        text = f.read()
    # Extract final val_bpb
    bpb_matches = re.findall(r'val_bpb:(\d+\.\d+)', text)
    if bpb_matches:
        results.append((name, float(bpb_matches[-1])))
    # Copy to Drive for persistence
    shutil.copy2(logfile, f'{DRIVE_DIR}/logs/{name}.txt')

results.sort(key=lambda x: x[1])
print(f"{'Experiment':<45} {'val_bpb':>10}")
print('-' * 57)
baseline_bpb = None
for name, bpb in results:
    if 'baseline' in name:
        baseline_bpb = bpb
    delta = f' ({bpb - baseline_bpb:+.4f})' if baseline_bpb and 'baseline' not in name else ''
    print(f"{name:<45} {bpb:>10.4f}{delta}")

# Save summary to Drive
with open(f'{DRIVE_DIR}/results/summary.txt', 'w') as f:
    f.write(f"{'Experiment':<45} {'val_bpb':>10}\n")
    f.write('-' * 57 + '\n')
    for name, bpb in results:
        delta = f' ({bpb - baseline_bpb:+.4f})' if baseline_bpb and 'baseline' not in name else ''
        f.write(f"{name:<45} {bpb:>10.4f}{delta}\n")
print(f'\nResults saved to {DRIVE_DIR}/results/summary.txt')